# 09 -- Apply Fuzzy Review

**Purpose**: Apply manual `match` / `new` decisions from `data/review_fuzzy_matches.csv`.
Run this notebook whenever a review file is present (generated by notebook 08 ingestion cells).

**Workflow**:
1. Check `data/review_fuzzy_matches.csv` exists and display pending rows
2. Apply decisions — `match` links existing prospect (no new row), `new` adds to `dim_rookie_prospect`
3. Archive review file as `review_fuzzy_matches.applied_YYYYMMDD.csv`

**After running**: Re-run the FantasyPros and WalterFootball ingestion cells in
`08_fact_rookie_rankings_seed.ipynb` to pick up newly added prospects and append
their rankings to `fact_rookie_rankings`.

**Tables updated**: `dim_rookie_prospect` (new rows), review file archived.

**Prerequisites**: `01_dim_rookie_prospect.ipynb`

In [ ]:
# ---- Setup: shared helpers + config from etl_helpers ------------------------
import pandas as pd
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
from etl_helpers import CFG, DATA, REVIEW, TODAY, clean_player_name, generate_player_key

REVIEW_PATH = REVIEW / "review_fuzzy_matches.csv"
ALIAS_PATH  = DATA / "dim_player_alias.parquet"   # persistent decisions (built by 08y)


def record_alias(review: pd.DataFrame) -> None:
    # Append every match/new decision to dim_player_alias so the matcher never
    # re-asks and ingest can attribute name-variants to the resolved player_key.
    rows = []
    for _, r in review.iterrows():
        action = str(r.get("action", "")).strip().lower()
        nc = r.get("new_name_clean")
        if pd.isna(nc) or not str(nc).strip():
            nc = clean_player_name(r.get("new_name", ""))
        pos = str(r.get("new_position", "")).upper().strip()
        if action == "match":
            pk = r.get("best_match_key")
        elif action == "new":
            pk = generate_player_key(r.get("new_name", ""), pos, str(r.get("new_school", "")))
        else:
            continue
        if pd.isna(pk) or not str(pk).strip():
            continue
        rows.append({"name_clean": str(nc), "position_raw": pos, "player_key": str(pk),
                     "decision": action, "source_example": r.get("source", ""),
                     "decided_date": TODAY})
    if not rows:
        return
    adf = pd.DataFrame(rows)
    if ALIAS_PATH.exists():
        adf = pd.concat([pd.read_parquet(ALIAS_PATH), adf], ignore_index=True)
    adf.drop_duplicates(subset=["name_clean", "position_raw"], keep="last", inplace=True)
    adf.to_parquet(ALIAS_PATH, index=False)
    print(f"dim_player_alias: {len(adf)} rows (+{len(rows)} from this apply)")

## 1 -- Check Review File

Displays all pending rows. Every row must have `action` set to `match` or `new` before applying.

| action | meaning |
|---|---|
| `match` | Player already exists in `dim_rookie_prospect` under the `best_match_name` — no new row needed |
| `new` | Player is genuinely new — add as a prospect using `new_name`, `new_position`, `new_school` |

In [2]:
if not REVIEW_PATH.exists():
    print(f"No review file found at {REVIEW_PATH}")
    print("Nothing to apply. Run notebook 08 ingestion cells to generate one.")
else:
    review = pd.read_csv(REVIEW_PATH)
    total   = len(review)
    pending = review["action"].isin(["", None]).sum() if "action" in review.columns else total
    print(f"Review file: {REVIEW_PATH}")
    print(f"  Total rows : {total}")
    print(f"  Pending    : {pending}  (action not yet filled)")
    print(f"  Ready      : {total - pending}")
    print()
    display(review[["new_name", "new_position", "new_school", "best_match_name", "fuzzy_score", "source", "action"]])

Review file: data\review_fuzzy_matches.csv
  Total rows : 67
  Pending    : 0  (action not yet filled)
  Ready      : 67



,new_name,new_position,new_school,best_match_name,fuzzy_score,source,action
0,Omar Cooper Jr.,WR,Indiana,omar cooper,88,KTC_Superflex,match
1,De'Zhaun Stribling,WR,Mississippi,de'zhaun-ryan stribling,88,KTC_Superflex,match
2,Eli Raridon,TE,Notre Dame,elijah raridon,88,KTC_Superflex,match
3,Omar Cooper Jr.,WR,Indiana,omar cooper,88,KTC_1QB,match
4,De'Zhaun Stribling,WR,Mississippi,de'zhaun-ryan stribling,88,KTC_1QB,match
...,...,...,...,...,...,...,...
62,Eli Raridon,TE,NaN,elijah raridon,88,DraftSharks,match
63,Jamarion Miller,RB,NaN,jam miller,80,DraftSharks,match
64,Cyrus Allen,WR,NaN,marcus allen,78,DraftSharks,new
65,CJ Williams,WR,NaN,damonic williams,74,DraftSharks,new


## 2 -- Apply Decisions

Runs `apply_review_decisions()`:
- `action=match` — player linked to existing `dim_rookie_prospect` row, no insert
- `action=new`   — player inserted as new prospect; position + school joined from transformer tables
- All other `action` values — skipped with a warning
- Review file archived as `review_fuzzy_matches.applied_YYYYMMDD.csv` on success

In [ ]:
def apply_review_decisions(
    review_path: Path = REVIEW_PATH,
) -> pd.DataFrame:
    # Apply manual match/new decisions from the review CSV.
    # action="match" -> player already exists, no new row; recorded in dim_player_alias.
    # action="new"   -> create a new dim_rookie_prospect row; recorded in dim_player_alias.
    # Archive to *.applied_YYYYMMDD.csv ONLY when every row's action is filled —
    # a blank action means "not yet reviewed", so a *.applied_ file is a reliable
    # tell that all rows were actioned.
    if not review_path.exists():
        print("No review file found. Nothing to apply.")
        return pd.read_parquet(DATA / "dim_rookie_prospect.parquet")

    with review_path.open("r", encoding="utf-8", newline="") as f:
        review = pd.read_csv(f)

    pending = int(review["action"].fillna("").astype(str).str.strip().eq("").sum())

    dim_rp  = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")
    pos_map = pd.read_parquet(DATA / "dim_position.parquet")
    sch_map = pd.read_parquet(DATA / "dim_school.parquet")

    new_rows, linked, skipped = [], 0, 0
    for _, row in review.iterrows():
        action = str(row.get("action", "")).strip().lower()
        if action == "match":
            linked += 1
        elif action == "new":
            pos_raw    = str(row.get("new_position", "")).upper().strip()
            school_raw = str(row.get("new_school", "")).strip()
            pos_match  = pos_map[pos_map["position_raw"] == pos_raw]
            sch_match  = sch_map[sch_map["school_raw"] == school_raw]
            new_rows.append({
                "player_key":        generate_player_key(row["new_name"], pos_raw, school_raw),
                "player_name":       row["new_name"],
                "player_name_clean": clean_player_name(row["new_name"]),
                "position_raw":      pos_raw,
                "position_detail":   pos_match["position_detail"].iloc[0] if len(pos_match) else None,
                "position_group":    pos_match["position_group"].iloc[0]  if len(pos_match) else None,
                "side_of_ball":      pos_match["side_of_ball"].iloc[0]    if len(pos_match) else None,
                "fantasy_relevant":  pos_match["fantasy_relevant"].iloc[0] if len(pos_match) else False,
                "school_raw":        school_raw,
                "school_canonical":  sch_match["school_canonical"].iloc[0] if len(sch_match) else school_raw,
                "conference":        sch_match["conference"].iloc[0]       if len(sch_match) else "Unknown",
                "height_inches": None, "weight": None,
                "pfr_id": None, "cfb_id": None, "gsis_id": pd.NA,
                "draft_year": CFG.draft_year,
                "source": row.get("source", "review"),
                "added_date": TODAY,
            })
        else:
            skipped += 1   # blank/unknown action -> left pending, not applied

    if new_rows:
        dim_rp = pd.concat([dim_rp, pd.DataFrame(new_rows)], ignore_index=True)
        dim_rp.drop_duplicates(subset=["player_key"], inplace=True)

    dim_rp.to_parquet(DATA / "dim_rookie_prospect.parquet", index=False)
    print(f"Review applied: {linked} matched, {len(new_rows)} new prospects added, {skipped} pending/skipped")
    print(f"dim_rookie_prospect total: {len(dim_rp)}")

    # Persist decisions so they are never re-asked and variants ingest correctly.
    record_alias(review)

    # Only archive a fully-actioned review (the *.applied_ tell). If rows are still
    # blank, keep the file active so the remaining decisions are visible.
    if pending:
        print(f"NOT archived — {pending} row(s) still have a blank action. "
              f"Fill them, then re-run to archive.")
        return dim_rp

    archive = review_path.with_suffix(f".applied_{TODAY.replace('-', '')}.csv")
    try:
        review_path.rename(archive)
        print(f"Review archived: {archive.name}")
    except PermissionError:
        review.to_csv(archive, index=False)
        try:
            pd.DataFrame(columns=review.columns).to_csv(review_path, index=False)
            print(f"Review archived: {archive.name} (original truncated to header-only)")
        except PermissionError:
            print(f"WARN: {review_path.name} is locked — close it in Excel and delete manually.")
            print(f"      Archive written to {archive.name}.")

    return dim_rp


dim_rookie_prospect = apply_review_decisions()

## 3 -- Verify & Next Steps

In [4]:
dim_rp = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")
print(f"dim_rookie_prospect: {len(dim_rp)} total prospects")
print()
print("Position breakdown:")
print(dim_rp["position_group"].value_counts().to_string())
print()
print("Sources contributing:")
print(dim_rp["source"].value_counts().to_string())
print()
print("=" * 60)
print("Next step: re-run ingestion cells in 08_fact_rookie_rankings_seed.ipynb")
print("  - FantasyPros run cell  (section 7)")
print("  - WalterFootball run cell (section 9)")
print("This picks up newly added prospects and appends their rankings to fact_rookie_rankings.")

dim_rookie_prospect: 393 total prospects

Position breakdown:
position_group
DL    71
WR    67
DB    61
OL    56
RB    42
TE    34
LB    28
QB    27
ST     7

Sources contributing:
source
nflverse_combine         319
FantasyPros_PPR           39
FantasyPros_Superflex     11
WalterFootball_WR          6
WalterFootball_DE          5
WalterFootball_CB          4
WalterFootball_S           3
WalterFootball_RB          2
WalterFootball_DT          2
WalterFootball_QB          1
WalterFootball_TE          1

Next step: re-run ingestion cells in 08_fact_rookie_rankings_seed.ipynb
  - FantasyPros run cell  (section 7)
  - WalterFootball run cell (section 9)
This picks up newly added prospects and appends their rankings to fact_rookie_rankings.
